# V13 selfplay — pillar3a + MCTS@200 on Colab L4

Generates the selfplay portion of the V13 corpus. Crisis mining runs locally on M5 (separate workflow — see "Local M5 crisis mining" section at the end).

## Inputs

- **Pillar3a** = `sharp_25_epoch_12.pt` (HISTORY 157). Local: `alphatrain/data/sharp_25_epoch_12.pt` → upload to Drive AS `sharp_25_epoch_12.pt`.
- **Value head pillar3a** = retrained on pillar3a backbone (HISTORY 158). Local: `alphatrain/data/value_head_sharp25_ep12.pt` (48 min M5 train). Upload to Drive AS `value_head_pillar3a.pt` (notebook reads under that name).

The retrained value head produces **+30% MCTS mean lift** over the OLD value_head_v12_v12targets (21,310 vs 16,340 on 100-seed cap=25K eval), with **floor cut from 7% to 1% <1000**. Using this combination for V13 generation produces dramatically sharper visit-distribution targets.

## Cap reasoning

`MAX_TURNS = 10000` — slight headroom over pillar3a P75. Going higher gives marginal late-game states at the cost of running 25% of games 3× longer. Better to spend that compute on more seeds.

## Workflow

1. **Step 0 (DONE locally)**: value_head_sharp25_ep12.pt produced.
2. **This notebook (Colab L4, ~6-12h)**: 1000 selfplay games × MCTS@200 × cap 10K → ~3-4M state-visit pairs.
3. **Local M5 (overnight, ~12-25h)**: crisis mining, ~4-8K seeds. Runs while Colab generates selfplay.
4. **Local M5 (~30 min)**: tensor build combining selfplay/ + crisis/ JSONs.

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — latest code (rebuilt now)
2. `sharp_25_epoch_12.pt` (rename to `sharp_25_epoch_12.pt` if not already — local file is exactly this)
3. **`value_head_pillar3a.pt`** — UPLOAD the local file `value_head_sharp25_ep12.pt` under this Drive name


In [ ]:
# ==== CONFIG — edit these before running ====
PILLAR3_NAME = 'sharp_25_epoch_12'   # CONFIRMED pillar3a (HISTORY 157)
MCTS_SIMS = 200             # 100 = cheap, 200 = balanced, 400 = expensive
MAX_TURNS = 10000           # slight headroom over sharp_50 P75 (~8K turns)
N_SELFPLAY = 1000           # more seeds = more state diversity vs deeper games
Q_WEIGHT = 2.0              # validated for value_head_v12 (HISTORY 139)
WORKERS = 16
BATCH_SIZE = 8              # MCTS leaf batch — HISTORY 115: >8 destroys quality
VALUE_HEAD_NAME = 'value_head_pillar3a.pt'   # retrained on pillar3a backbone (HISTORY 158)

DRIVE_OUT_DIR = '/content/drive/MyDrive/alphatrain/v13'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/{PILLAR3_NAME}.pt',
            f'/content/alphatrain/data/pillar3.pt')
shutil.copy(f'{DRIVE}/{VALUE_HEAD_NAME}',
            f'/content/alphatrain/data/value_head.pt')
print(f'pillar3: {os.path.getsize("/content/alphatrain/data/pillar3.pt")/1e6:.0f} MB')
print(f'value head: {os.path.getsize("/content/alphatrain/data/value_head.pt")/1e6:.1f} MB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {g.total_memory/1e9:.1f} GB')

# Sanity: pillar3 should be substantially stronger than pillar2y2/2z.
# Quick policy-only eval on 50 seeds confirms checkpoint loaded right.
%cd /content
!python -m alphatrain.scripts.eval_parallel \
    --model alphatrain/data/pillar3.pt \
    --policy-only --device cuda --workers 8 \
    --max-turns 20000 --seeds $(seq 0 49) 2>&1 | tail -10

## Phase 1: Self-play (~6-12h)

Generate ~500 games with pillar3 + MCTS@200. Output: per-game JSON with state snapshots + MCTS visit distributions for the top-15 actions per state.

In [ ]:
import os
os.makedirs(f'{DRIVE_OUT_DIR}/selfplay', exist_ok=True)

%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.scripts.selfplay \
    --model alphatrain/data/pillar3.pt \
    --value-head-path alphatrain/data/value_head.pt \
    --simulations {MCTS_SIMS} \
    --top-k 30 --batch-size {BATCH_SIZE} \
    --q-weight {Q_WEIGHT} \
    --temperature-moves 15 \
    --dirichlet-alpha 0.3 --dirichlet-weight 0.25 \
    --max-turns {MAX_TURNS} \
    --num-games {N_SELFPLAY} \
    --workers {WORKERS} \
    --device cuda --compile \
    --save-dir {DRIVE_OUT_DIR}/selfplay

## Crisis mining and tensor build — run LOCALLY on M5

Crisis mining doesn't benefit from Colab speed and consumes the 24h kill clock unnecessarily. Run it on M5 locally while this notebook is generating selfplay games on L4.

### Step 0 (run BEFORE this notebook): retrain value head

```bash
# On M5, ~1-2h with V11 corpus
python -m alphatrain.scripts.train_value_head \
    --backbone alphatrain/data/pillar3a.pt \
    --tensor-file alphatrain/data/v12_pillar2z.pt \
    --output alphatrain/data/value_head_pillar3a.pt \
    --epochs 5 --batch-size 8192 --lr 1e-3
```

Then upload `value_head_pillar3a.pt` to Drive before launching this notebook.

### Crisis mining (run during/after selfplay)

```bash
# On M5, ~12-25h depending on N_CRISIS_SEEDS
N_CRISIS_SEEDS=4000   # or 8000 for full corpus, ~25h
python -m alphatrain.scripts.crisis_mining \
    --model alphatrain/data/pillar3a.pt \
    --value-head-path alphatrain/data/value_head_pillar3a.pt \
    --seed-start 800000 --seed-end $((800000 + N_CRISIS_SEEDS)) \
    --recovery-turns 15 --recovery-sims 600 \
    --prevention-turns 30 --prevention-sims 600 \
    --continue-turns 500 \
    --policy-max-turns 8000 \
    --workers 16 --device mps --batch-size 8 \
    --q-weight 2.0 \
    --save-dir data/crisis_v13
```

Note: no `--max-turns` flag — `--continue-turns 500` is the binding cap on replay length, `--policy-max-turns 8000` caps the probe. `--max-turns` is defensive padding that doesn't drive behavior at these settings.

### Tensor build (after both selfplay and crisis done)

```bash
# Download Drive selfplay/ to local data/selfplay_v13/, then:
python -m alphatrain.scripts.build_expert_v2_tensor \
    --selfplay-dir data/selfplay_v13 \
    --crisis-dir data/crisis_v13 \
    --policy-only-data \
    --output alphatrain/data/v13_pillar3a.pt
```

## After V13 lands

1. **Verify V13 target entropy** vs V12:
   ```
   python -m alphatrain.scripts.analyze_v12_target_entropy \
       --tensor-file alphatrain/data/v13_pillar3a.pt --n-samples 5000
   ```
   Expected: V13 targets at T=1.0 should have top1 mean > 0.30 (sharper than V12's 0.26), because pillar3a+MCTS produces more decisive visit distributions. If true, sharpening in pillar4a training may need less temperature.

2. **Train pillar4a** (~17 epochs, ~12h Colab G4):
   ```
   python -m alphatrain.train_path_b \
       --tensor-file alphatrain/data/v13_pillar3a.pt --amp --compile \
       --resume alphatrain/data/pillar3a.pt --warm-start \
       --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
       --target-temperature 0.5
   ```
   (Color + dihedral augmentation are default-on; no flags needed.)
   Pick T=0.7 if V13 entropy check shows already-sharper targets.

3. **Evaluate pillar4a** policy + MCTS@200. Decision criterion: ≥+30% over pillar3a mean.
